# Insurance Fraud Detection

## Module 1: Data Understanding and Cleaning

Objectives:
- Understand the insurance claims dataset.
- Inspect missing values and duplicates.
- Validate fraud labels.
- Save a cleaned dataset.

In [1]:
from src.data_processing.cleaning import (
    RAW_DATA_PATH,
    CLEANED_DATA_PATH,
    load_data,
    inspect_data,
    clean_data,
    save_cleaned_data,
)

In [2]:
raw_data = load_data(RAW_DATA_PATH)

print("Dataset shape:", raw_data.shape)
display(raw_data.head())

inspect_data(raw_data)

Dataset shape: (1000, 39)


,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,521585,17-10-2014,OH,250/500,1000,1406.91,0,466132,...,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y
1,228,42,342868,27-06-2006,IN,250/500,2000,1197.22,5000000,468176,...,0,NaN,5070,780,780,3510,Mercedes,E400,2007,Y
2,134,29,687698,06-09-2000,OH,100/300,2000,1413.14,5000000,430632,...,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N
3,256,41,227811,25-05-1990,IL,250/500,2000,1415.74,6000000,608117,...,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y
4,228,44,367455,06-06-2014,IL,500/1000,1000,1583.91,6000000,610706,...,1,NO,6500,1300,650,4550,Accura,RSX,2009,N



--- Dataset overview ---
Rows: 1000
Columns: 39

Column data types:
months_as_customer               int64
age                              int64
policy_number                    int64
policy_bind_date                object
policy_state                    object
policy_csl                      object
policy_deductable                int64
policy_annual_premium          float64
umbrella_limit                   int64
insured_zip                      int64
insured_sex                     object
insured_education_level         object
insured_occupation              object
insured_hobbies                 object
insured_relationship            object
capital-gains                    int64
capital-loss                     int64
incident_date                   object
incident_type                   object
collision_type                  object
incident_severity               object
authorities_contacted           object
incident_state                  object
incident_city                   ob

In [3]:
cleaned_data = clean_data(raw_data)

display(cleaned_data.head())

save_cleaned_data(cleaned_data, CLEANED_DATA_PATH)


--- Cleaning summary ---
Exact duplicate rows removed: 0
Rows removed for missing fraud labels: 0
Rows remaining: 1000
Columns remaining: 39

Missing values remaining by column:
collision_type             178
authorities_contacted       91
property_damage            360
police_report_available    343

Validated fraud label counts:
fraud_reported
N    753
Y    247


,months_as_customer,age,policy_number,policy_bind_date,policy_state,policy_csl,policy_deductable,policy_annual_premium,umbrella_limit,insured_zip,...,witnesses,police_report_available,total_claim_amount,injury_claim,property_claim,vehicle_claim,auto_make,auto_model,auto_year,fraud_reported
0,328,48,521585,17-10-2014,OH,250/500,1000,1406.91,0,466132,...,2,YES,71610,6510,13020,52080,Saab,92x,2004,Y
1,228,42,342868,27-06-2006,IN,250/500,2000,1197.22,5000000,468176,...,0,NaN,5070,780,780,3510,Mercedes,E400,2007,Y
2,134,29,687698,06-09-2000,OH,100/300,2000,1413.14,5000000,430632,...,3,NO,34650,7700,3850,23100,Dodge,RAM,2007,N
3,256,41,227811,25-05-1990,IL,250/500,2000,1415.74,6000000,608117,...,2,NO,63400,6340,6340,50720,Chevrolet,Tahoe,2014,Y
4,228,44,367455,06-06-2014,IL,500/1000,1000,1583.91,6000000,610706,...,1,NO,6500,1300,650,4550,Accura,RSX,2009,N



Cleaned dataset saved to: D:\AI\ML\Insurance_fraud_detection\data\insurance_claims_cleaned.csv


In [4]:
print("Cleaned shape:", cleaned_data.shape)
print("\nFraud distribution:")
display(cleaned_data["fraud_reported"].value_counts())

print("\nMissing values:")
display(cleaned_data.isna().sum().sort_values(ascending=False).head(10))

assert not cleaned_data.duplicated().any()
assert cleaned_data["fraud_reported"].isin(["Y", "N"]).all()

print("PASS: Notebook validation")

Cleaned shape: (1000, 39)

Fraud distribution:


fraud_reported
N    753
Y    247
Name: count, dtype: int64


Missing values:


property_damage            360
police_report_available    343
collision_type             178
authorities_contacted       91
months_as_customer           0
age                          0
policy_number                0
policy_deductable            0
policy_csl                   0
policy_state                 0
dtype: int64

PASS: Notebook validation


## Module 2: Preprocessing and Leakage Prevention

- Separate features and target.
- Split training and test datasets.
- Fit preprocessing on training data only.
- Impute missing values.
- Encode categorical variables.
- Scale numerical variables.

In [1]:
from src.data_processing.preprocessing import (
    INPUT_PATH,
    load_cleaned_data,
    prepare_features_and_target,
    build_preprocessing_pipeline,
)

from sklearn.model_selection import train_test_split

data = load_cleaned_data(INPUT_PATH)
X, y = prepare_features_and_target(data)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

preprocessing = build_preprocessing_pipeline(X_train)

X_train_processed = preprocessing.fit_transform(X_train)
X_test_processed = preprocessing.transform(X_test)

print("Training rows:", len(X_train_processed))
print("Test rows:", len(X_test_processed))
print("Training fraud rate:", round(y_train.mean(), 3))
print("Test fraud rate:", round(y_test.mean(), 3))


--- Feature preparation ---
Excluded columns: ['policy_number', 'insured_zip', 'incident_location', 'policy_bind_date', 'incident_date', 'authorities_contacted', 'police_report_available', 'incident_severity']
Feature count: 30
Target counts:
fraud_reported
Legitimate (0)    753
Fraud (1)         247
Name: count, dtype: int64
Training rows: 800
Test rows: 200
Training fraud rate: 0.248
Test fraud rate: 0.245


## Module 3: Model Training

- Logistic Regression baseline
- Random Forest comparison
- Five-fold stratified cross-validation
- Hyperparameter tuning
- Model selection using cross-validation F1-score

In [2]:
from src.modeling.training import load_training_data, train_and_compare_models

X_train, y_train = load_training_data()

best_model, best_name, best_score, results = train_and_compare_models(
    X_train, y_train
)

print("Selected model:", best_name)
print("Cross-validation F1:", round(best_score, 4))
display(results)


Training: Logistic Regression
Best cross-validation F1: 0.4593
Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}

Training: Random Forest
Best cross-validation F1: 0.3337
Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 10, 'classifier__n_estimators': 100}
Selected model: Logistic Regression
Cross-validation F1: 0.4593


,model,cv_f1,best_parameters
0,Logistic Regression,0.459260,"{'classifier__C': 0.1, 'classifier__class_weig..."
1,Random Forest,0.333694,"{'classifier__class_weight': 'balanced', 'clas..."


Module 4: Model Evaluation
--

In [3]:
from src.modeling.evaluation import load_test_data, evaluate_model
import joblib

X_test, y_test = load_test_data()
model = joblib.load("data/best_fraud_model.joblib")

metrics, report, matrix = evaluate_model(model, X_test, y_test)

print("Test metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

display(report.round(4))
display(matrix)

Test metrics:
accuracy: 0.6200
precision: 0.3247
recall: 0.5102
f1_score: 0.3968
roc_auc: 0.6133


,precision,recall,f1-score,support
Legitimate,0.8049,0.6556,0.7226,151.00
Fraud,0.3247,0.5102,0.3968,49.00
accuracy,0.6200,0.6200,0.6200,0.62
macro avg,0.5648,0.5829,0.5597,200.00
weighted avg,0.6872,0.6200,0.6428,200.00


array([[99, 52],
       [24, 25]])

## Module 5: Prediction

The saved model can predict new insurance claims without retraining.

Input: claim features matching the original training schema.

Output: predicted class, fraud label and estimated fraud probability.

In [4]:
import pandas as pd
from src.modeling.prediction import load_model, predict_claim

# Load a sample claim's features, excluding the target.
claim = pd.read_csv("data/new_claim.csv")

model = load_model()
prediction = predict_claim(model, claim)

display(prediction)

,predicted_class,prediction,fraud_probability
0,0,Legitimate,0.251737
